# 유형 03 — 완전탐색 (Brute Force)

> **대상**: 정렬·자료구조를 익힌 단계.
> **목표**: "가능한 모든 경우를 빠짐없이 만들어 확인한다"를 코드로 옮기기. 핵심 도구는 **`itertools`**와 **재귀**.

완전탐색은 **"경우의 수가 작을 때, 전부 다 해보고 답을 고르는"** 전략입니다. 똑똑한 알고리즘이 떠오르지 않아도, 경우의 수가 작으면(보통 수백만 이하) 그냥 다 해보는 게 정답입니다.

## 0) 먼저 판단 — 완전탐색을 써도 되나?

"모든 경우를 다 만든다"는 경우의 수가 폭발하면 못 씁니다. 대략적인 감:

- 순열 `n!` → n이 10이면 약 360만 (OK), 13이면 60억 (위험)
- 부분집합 `2^n` → n이 20이면 약 100만 (OK), 30이면 10억 (위험)

> 규칙: **경우의 수가 약 1000만 이하면 완전탐색을 의심**하세요. 입력 크기가 작게 제한돼 있으면(예: `n ≤ 8`) 출제자가 "다 해봐라"라고 알려주는 신호입니다.

## 1) itertools — 경우의 수 만들기 4종

```python
from itertools import permutations, combinations, product

nums = [1, 2, 3]

# 순열: 순서가 다르면 다른 것 (뽑아서 나열)
list(permutations(nums, 2))   # [(1,2),(1,3),(2,1),(2,3),(3,1),(3,2)]

# 조합: 순서 무관, 뽑기만
list(combinations(nums, 2))   # [(1,2),(1,3),(2,3)]

# 곱집합: 각 자리에서 하나씩 (중복 허용 선택)
list(product([0, 1], repeat=2))  # [(0,0),(0,1),(1,0),(1,1)]
```

| 도구 | 언제 | 예 |
|------|------|----|
| `permutations(arr, r)` | "순서가 중요", "나열/배치" | 숫자로 만들 수 있는 모든 N자리 수 |
| `combinations(arr, r)` | "순서 무관", "r개 고르기" | 후보 중 2명 뽑는 모든 경우 |
| `product(arr, repeat=n)` | "각 칸에 독립적으로 선택" | n자리 비밀번호 모든 조합, ON/OFF 스위치 |

> 핵심: 지문을 보고 **"순서가 의미 있나? 같은 걸 또 뽑나?"** 두 질문으로 셋 중 하나를 고릅니다.

In [1]:
from itertools import permutations, combinations, product

nums = [1, 2, 3]

print(list(permutations(nums, 2)))
print(list(combinations(nums, 2)))
print(list(product([0, 1], repeat=2)))

[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
[(1, 2), (1, 3), (2, 3)]
[(0, 0), (0, 1), (1, 0), (1, 1)]


## 2) 재귀로 모든 경우 만들기 (백트래킹의 씨앗)

`itertools`로 안 떨어지는 복잡한 경우는 재귀로 직접 만듭니다.

```python
# 0/1을 n자리로 만드는 모든 경우 (product와 같은 결과)
def build(n, cur, results):
    if len(cur) == n:            # 다 채웠으면 하나 완성
        results.append(cur[:])
        return
    for choice in (0, 1):        # 이 자리에서 가능한 선택
        cur.append(choice)       # 선택하고
        build(n, cur, results)   # 다음 자리로
        cur.pop()                # 되돌리기 (백트래킹)

results = []
build(2, [], results)
# [[0,0],[0,1],[1,0],[1,1]]
```

> 패턴: **선택 → 재귀 → 되돌리기**. 이 3박자가 완전탐색/백트래킹의 뼈대입니다. 05(DFS/BFS)에서 다시 만납니다.


In [3]:
def build(n, cur, results):
    if len(cur) == n:
        results.append(cur[:])
        return
    for choice in (0, 1):
        cur.append(choice)
        build(n, cur, results)
        cur.pop()

results = []
build(2, [], results)
print(results)

[[0, 0], [0, 1], [1, 0], [1, 1]]


## 대표 문제 풀이 — 모의고사 (프로그래머스 42840)

> 수포자 3명이 각자 정해진 패턴으로 답을 찍는다. `answers`(정답 배열)에 대해 **가장 많이 맞힌 사람(들)**의 번호를 오름차순으로 반환하라.

**① 신호**: "정해진 패턴대로", "가장 많이 맞힌 사람" — 경우가 3명뿐, 전부 채점
**② 패턴**: 완전탐색 (세 사람 × 모든 문제를 다 비교)

```python
def solution(answers):
    # 세 수포자의 찍기 패턴 (반복됨)
    patterns = [
        [1, 2, 3, 4, 5],
        [2, 1, 2, 3, 2, 4, 2, 5],
        [3, 3, 1, 1, 2, 2, 4, 4, 5, 5],
    ]
    scores = [0, 0, 0]
    for i, ans in enumerate(answers):
        for p in range(3):
            # 패턴은 반복되므로 % 로 자리 맞춤
            if ans == patterns[p][i % len(patterns[p])]:
                scores[p] += 1

    highest = max(scores)
    # 최고점인 사람(들)을 1-base 번호로, 오름차순
    return [i + 1 for i in range(3) if scores[i] == highest]
```

**검증:**

```python
print(solution([1, 2, 3, 4, 5]))        # [1]      (1번이 다 맞힘)
print(solution([1, 3, 2, 4, 2]))        # [1, 2, 3] (셋이 동점)

assert solution([1, 2, 3, 4, 5]) == [1]
assert solution([1, 3, 2, 4, 2]) == [1, 2, 3]
print("통과 ✅")
```

> 가르칠 포인트:
> 1. **패턴 반복**은 `i % len(pattern)`으로 자리를 맞춥니다 (문제 수가 패턴보다 길어도 OK).
> 2. **동점 처리** — `max`로 최고점을 구한 뒤, 그 점수와 같은 사람을 모두 모읍니다. "최고가 여러 명일 수 있다"를 놓치지 않기.


In [4]:
def solution(answers):
    patterns = [
        [1, 2, 3, 4, 5],
        [2, 1, 2, 3, 2, 4, 2, 5],
        [3, 3, 1, 1, 2, 2, 4, 4, 5, 5],
    ]
    scores = [0, 0, 0]
    for i, ans in enumerate(answers):
        for p in range(3):
            if ans == patterns[p][i % len(patterns[p])]:
                scores[p] += 1

    highest = max(scores)

    return [i + 1 for i in range(3) if scores[i] == highest]

In [5]:
print(solution([1, 2, 3, 4, 5]))        # [1]      (1번이 다 맞힘)
print(solution([1, 3, 2, 4, 2]))        # [1, 2, 3] (셋이 동점)

assert solution([1, 2, 3, 4, 5]) == [1]
assert solution([1, 3, 2, 4, 2]) == [1, 2, 3]
print("통과 ✅")

[1]
[1, 2, 3]
통과 ✅


## 직접 풀어보기 (연습)

힌트를 보고 `TODO`를 채운 뒤 검증 셀을 실행하세요. 완전탐색은 **"모든 경우를 어떻게 빠짐없이 만들까(itertools/재귀)?"**부터 정합니다.

### 연습 1 — 두 수의 합으로 가능한 값의 개수

> 정수 리스트 `nums`에서 서로 다른 두 수를 뽑아 더했을 때, 나올 수 있는 **서로 다른 합**의 개수를 구하라.
> **힌트**: `combinations(nums, 2)`로 모든 두 수 쌍 → 합을 `set`에 모아 개수.

```python
def solution(nums):
    # TODO: 직접 작성
    pass

# 검증
assert solution([2, 1, 3, 4, 1]) == 6   # 가능한 합: {3,4,5,2,6,7} → 6개
assert solution([5, 0, 2, 7]) == 5      # {5,7,12,2,9} → 5개
print("통과 ✅")
```

### 연습 2 — 모든 N자리 수 만들기

> 서로 다른 한 자리 숫자 리스트 `digits`로 만들 수 있는 모든 **두 자리 수**의 합을 구하라. (같은 숫자를 두 번 쓰지 않음)
> **힌트**: `permutations(digits, 2)` → 각 `(a, b)`를 `a*10 + b`로 만들어 합산.

```python
def solution(digits):
    # TODO: 직접 작성
    pass

# 검증
assert solution([1, 2]) == 33      # 12 + 21
assert solution([1, 2, 3]) == 264  # 12+13+21+23+31+32
print("통과 ✅")
```

### 연습 3 — 소수 찾기 (프로그래머스 42839, 도전)

> 숫자가 적힌 종이조각 문자열 `numbers`의 각 자리를 조합해 만들 수 있는 수 중, **소수의 개수**를 구하라. (예: `"17"` → 1,7,17,71 중 소수는 7,17,71 → 3개)
> **힌트**: 길이 1~len 각각에 대해 `permutations`로 모든 수를 만들고 `set`에 모음(중복·앞자리 0 제거) → 각 수가 소수인지 판별해 카운트.

```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

def solution(numbers):
    # TODO: is_prime을 활용해 직접 작성
    pass

# 검증
assert solution("17") == 3     # 7, 17, 71
assert solution("011") == 2    # 11, 101
print("통과 ✅")
```
